In [0]:
df = spark.table("samples.bakehouse.sales_transactions")
display(df.head(10))

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4872480716880043
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5287105980593305
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
df.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



In [0]:
import time
import pyspark.sql.functions as F

start = time.time()

df.groupBy("franchiseID").agg(
    F.sum("quantity").alias("total_qty"),
    F.avg("totalPrice").alias("avg_totalPrice")
).collect()

display(df.head(10))

end = time.time()

print("Execution time:", end - start)

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4872480716880043
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5287105980593305
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


Execution time: 1.3763198852539062


In [0]:
df.groupBy("franchiseID").sum("quantity").explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- == Initial Plan ==
   PhotonResultStage (4)
   +- PhotonColumnarToRow (3)
      +- PhotonGroupingAgg (2)
         +- PhotonScan parquet samples.bakehouse.sales_transactions (1)


(1) PhotonScan parquet samples.bakehouse.sales_transactions
Output [2]: [franchiseID#14598L, quantity#14601L]
Location: PreparedDeltaFileIndex [s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b622f329-0f39-444f-9dbb-5733ba3e2021/tables/e922f6b4-1ca7-495e-b661-a5e73dc5f5d5]
ReadSchema: struct<franchiseID:bigint,quantity:bigint>

(2) PhotonGroupingAgg
Input [2]: [franchiseID#14598L, quantity#14601L]
Arguments: [franchiseID#14598L], [sum(quantity#14601L)], [sum(quantity)#14606L], [franchiseID#14598L, sum(quantity)#14606L AS sum(quantity)#14607L], true

(3) PhotonColumnarToRow
Input [2]: [franchiseID#14598L, sum(quantity)#14607L]

(4) PhotonResultStage
Input [2]: [franchiseID#14598L, sum(quantity)#14607L]

(5) AdaptiveSparkPlan
Output [2]: [franchiseID#1

### 🔎 Execution Plan Observations

The physical plan shows that Spark is using **Photon** for execution, which means the query is optimized and fully supported by Databricks’ high-performance engine. Only the required columns (`franchiseID`, `quantity`) were scanned, confirming **column pruning** is working properly `Less columns → Less I/O → Faster`. The `PhotonGroupingAgg` step indicates aggregation with a shuffle phase which is expensive.

However, optimizer statistics are marked as **missing**, meaning Spark doesn’t have full metadata for better cost-based optimization.

---


### Reducing Unnecessary Actions:

In [0]:
df.count()
df.show(3)
df.groupBy("product").sum("totalPrice").show(5)

+-------------+----------+-----------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|transactionID|customerID|franchiseID|            dateTime|             product|quantity|unitPrice|totalPrice|paymentMethod|      cardNumber|
+-------------+----------+-----------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|      1002961|   2000253|    3000047|2024-05-14 12:17:...|  Golden Gate Ginger|       8|        3|        24|         amex| 378154478982993|
|      1003007|   2000226|    3000047|2024-05-10 23:10:...|Austin Almond Bis...|      36|        3|       108|   mastercard|2244626981238094|
|      1003017|   2000108|    3000047|2024-05-16 16:34:...|Austin Almond Bis...|      40|        3|       120|   mastercard|2490570234487424|
+-------------+----------+-----------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
only s

#### Here, In the above code, Each is a separate Spark job that reads the whole data and recomputes the same thing repeatedly. By doing the below, only single job is triggered which is efficient

In [0]:
result = df.groupBy("product").sum("totalPrice")
result.show(5)

+--------------------+---------------+
|             product|sum(totalPrice)|
+--------------------+---------------+
|  Golden Gate Ginger|          11595|
|Austin Almond Bis...|          11148|
|         Pearly Pies|          10785|
|       Tokyo Tidbits|          10986|
|     Outback Oatmeal|          11199|
+--------------------+---------------+
only showing top 5 rows


## Compare Runtime Again After Optimization
---

In [0]:
start_opt = time.time()

optimized_df = df.select("product", "totalPrice")
optimized_df.groupBy("product").sum("totalPrice").show(10)

end_opt = time.time()
print("Execution time (after optimization):", end_opt - start_opt)

+--------------------+---------------+
|             product|sum(totalPrice)|
+--------------------+---------------+
|  Golden Gate Ginger|          11595|
|Austin Almond Bis...|          11148|
|         Pearly Pies|          10785|
|       Tokyo Tidbits|          10986|
|     Outback Oatmeal|          11199|
|       Orchard Oasis|          10758|
+--------------------+---------------+

Execution time (after optimization): 0.8013095855712891


In [0]:
print(f"This optimized task is {(end - start) - (end_opt - start_opt)} times faster")

This optimized task is 0.5750102996826172 times faster


--- 

# Cost Saving Ideas:
## 🔥 Cost Optimization Observations

### 1️⃣ Column Pruning

Selected only required columns (product, totalPrice)
→ Reduced I/O and memory usage.

> Note: Sparks does this automatically internally but its best to write explicitly

### 2️⃣ Early Filtering

Filtered dataset before aggregation
→ Reduced shuffle size

### 3️⃣ Reduced Actions

Avoided unnecessary .count() and .show()
→ Reduced number of Spark jobs

### 4️⃣ Avoided Recomputations

Stored transformation in variable before writing
→ Prevented duplicate execution